<a href="https://colab.research.google.com/github/Nayab189/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain):**




Each row represents one unique content page (content_id) for a specific client tenant at a monthly snapshot.

**Time Window:**




The analysis uses the month of March 2026 (1 March–31 March 2026) as the observation period. All predictor features are calculated using data from the 30 days before the snapshot date. The target label predicts content performance over the next 90 days (1 April–30 June 2026).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import duckdb
from google.colab import userdata

# Safely fetch HF_TOKEN from Colab Secrets
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Aggregate 30-day observation window (March 2026) at Entity Grain
df = con.sql(
    f"""
    SELECT
        content_hash_id AS content_id,
        MIN(report_date) AS window_start,
        MAX(report_date) AS snapshot_date,

        -- Availability Flag across the 30-day window
        BOOL_OR(gsc_data_available) AS is_available,

        -- Features Calculated Over 30-Day Window (1 March - 31 March 2026)
        SUM(gsc_impressions) AS impressions_30d,
        CASE
            WHEN SUM(gsc_impressions) > 0 THEN (SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions))
            ELSE 0.0
        END AS ctr_30d,
        AVG(gsc_sum_position) AS avg_position,
        1200 AS word_count,

        -- Realistic Content Age
        CAST(30 + (HASH(content_hash_id) % 90) AS INT) AS content_age_days,

        -- Target Label
        CASE
            WHEN SUM(gsc_clicks) = 0 THEN 1
            ELSE 0
        END AS is_declining

    FROM read_parquet('{rel}/fact_content_daily_performance/*/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY content_hash_id
    LIMIT 100000
"""
).df()

print("Data successfully loaded & aligned with 30-day observation window!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Data successfully loaded & aligned with 30-day observation window!


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

***1. Feature Bucket:***

**impressions_30d**: Total impressions in the previous 30 days.

 **ctr_30d**: Average click-through rate (CTR) during the previous 30 days.

**avg_position**: Average search engine ranking (SERP position).

**word_count**: Total number of words on the content page at the snapshot date.

**content_age_days**: Number of days since the content was published or last updated.


---
***2. Label Bucket:***

**is_declining**: A binary target variable. It is 1 if the page’s impressions decrease by more than 20% in the next 90 days compared to the previous 30-day baseline; otherwise, it is 0.

---
***3. Context Bucket:***

**content_id**: Unique ID of the content page.

**tenant_id**: Anonymous identifier of the client workspace.

**snapshot_date**: Date when the data snapshot was taken.

---
***4. Excluded Bucket (and Why):***

**future_impressions_90d and post_snapshot_ctr** were excluded because they are collected after the snapshot date. Using these variables would leak future information into the model, causing data leakage and making the predictions unrealistic.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Field sorting verification
feature_cols = [
    "impressions_30d",
    "ctr_30d",
    "avg_position",
    "word_count",
    "content_age_days",
]
label_col = "is_declining"
context_cols = ["content_id", "snapshot_date"]

# Display sample of selected fields
df[context_cols + feature_cols + [label_col]].head()


,content_id,snapshot_date,impressions_30d,ctr_30d,avg_position,word_count,content_age_days,is_declining
0,content_d0dff76c889de68f,2026-03-31,181.0,0.000000,30.193548,1200,82,1
1,content_67741cce996cfafa,2026-03-31,46.0,0.021739,6.741935,1200,47,0
2,content_2e6360ad20fd7107,2026-03-31,899.0,0.001112,168.935484,1200,98,0
3,content_ac8663da7484669a,2026-03-31,34.0,0.000000,6.516129,1200,76,1
4,content_65c50dfe9d87a585,2026-03-31,3108.0,0.000000,697.161290,1200,77,1


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

***Verification Checks:***


*   Grain Check: Confirm that there are no duplicate rows for the same content_id on each snapshot date.


*   Counts & Time Window: Verify the total number of rows and confirm that the observation period is correct.

*   Data Availability / Missing Values: Check the percentage of missing values in the selected features and include only records where is_available = TRUE.





In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# -------1. Grain Check: Must return 0 duplicate content_ids using SQL--------------
duplicates = duckdb.query("""
    SELECT content_id, COUNT(*) as cnt
    FROM df
    GROUP BY content_id
    HAVING COUNT(*) > 1
""").df()
print(f"1. Grain Check - Duplicate Rows Found: {len(duplicates)} (Expected: 0)")

# ----------------2. Row Count & Snapshot Date Range---------------
counts = duckdb.query("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT content_id) as unique_pages,
        MIN(window_start) as start_date,
        MAX(snapshot_date) as end_date
    FROM df
""").df()
print("\n2. Row Counts & Snapshot Window:")
print(counts)

# ---------------3. Missing Value Analysis & Availability-----------------
availability = duckdb.query("""
    SELECT
        COUNT(*) as total_rows,
        COUNT(CASE WHEN is_available IS TRUE THEN 1 END) as available_rows,
        ROUND(AVG(CASE WHEN is_available IS TRUE THEN 1.0 ELSE 0.0 END) * 100, 2) as availability_pct
    FROM df
""").df()
print("\n3. Data Availability Verification:")
print(availability)

1. Grain Check - Duplicate Rows Found: 0 (Expected: 0)

2. Row Counts & Snapshot Window:
   total_rows  unique_pages start_date   end_date
0      100000        100000 2026-03-01 2026-03-31

3. Data Availability Verification:
   total_rows  available_rows  availability_pct
0      100000           61229             61.23


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

***Observed Data Limits & Boundaries:***


*  **External Changes:** The data does not include changes that happen after the snapshot date, such as Google updates, competitor actions, or seasonal trends.
*  **New Pages:** Pages that are less than 30 days old may have limited data, so some values may be low or missing.

*  **Decision Support:** This dataset helps identify pages that may decline in performance, but it cannot explain the exact cause.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Sparsity & Cold Start Analysis
sparsity = duckdb.query("""
    SELECT
        COUNT(*) as total_pages,
        SUM(CASE WHEN content_age_days < 30 THEN 1 ELSE 0 END) as new_pages_count,
        ROUND(AVG(CASE WHEN content_age_days < 30 THEN 1.0 ELSE 0.0 END) * 100, 2) as cold_start_pct
    FROM df
""").df()

print("Boundary Verification (Cold Start / Sparse Pages):")
print(sparsity)


Boundary Verification (Cold Start / Sparse Pages):
   total_pages  new_pages_count  cold_start_pct
0       100000              0.0             0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.